# 00 · Data journey

## Purpose
Follow three recordings through every stage of the data layer, so each transformation is visible end to end:
folder → labels → raw CSV → signals → shaft speed → quality numbers → `.npy` file → `metadata.csv` row → splits.
Part B marks where Stage 2 (windows, features, spectrograms) will continue the journey.

## Inputs
The MAFAULDA CSV tree, `data/raw_npy/`, the committed `metadata.csv` and `configs/base.yaml`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tools, the settings, the data folder location and <code>metadata.csv</code>, and prints the data folder and a fingerprint (hash) of the preprocessing settings.</div>

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from ml.config import config_sha256, load_config
from ml.labels import parse_labels
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, data_dir, data_root
from ml.preprocess import choose_speed, load_csv, quality_stats, speed_candidates, tacho_intervals
from ml.splits import split_columns

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
root = data_root(cfg)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"data root {root}")
print(f"preprocess config sha256 {config_sha256(pre)[:16]}…")

## Part A · Three recordings

### Choosing them
One healthy machine and two faults, all at about 30 Hz shaft speed so their signals are comparable: normal,
imbalance with a 20 g mass, and an outer-race bearing fault under the underhang bearing with no added mass.
All three are in the closed-set **train** split and carry no quality flags.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Picks, for each of the three classes, the training recording closest to 30 Hz that has no quality flags, and lists the three chosen files.</div>

In [ ]:
def pick(class10: str, severity: float) -> pd.Series:
    c = meta[(meta.class10 == class10) & (meta.severity == severity) & (meta.split_closed == "train")
             & meta.quality_flags.isna()]
    return c.loc[(c.fr_measured_hz - 30).abs().idxmin()]

picks = [pick("normal", 0), pick("imbalance", 20), pick("underhang_outer", 0)]
pd.DataFrame(picks)[["rec_id", "path", "fr_measured_hz", "split_closed"]]

### Step 1 · Folder path → labels
The folder layout is the only source of labels. `ml.labels.parse_labels` reads the class from the folders and the
nominal shaft speed from the file name. For bearing faults the `g` folder is extra imbalance mass, not damage size.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Turns each file's folder path into its labels (fault family, type, severity, the 10-class and 6-class labels, and the nominal speed from the file name) and shows them side by side.</div>

In [ ]:
labels = {r.rec_id: parse_labels(r.path) for r in picks}
pd.DataFrame({rid: lab.__dict__ for rid, lab in labels.items()})

### Step 2 · The raw CSV
Each file is plain text: 250,000 rows (5 s at 50 kHz), 8 columns, no header. These are the first rows, exactly as
stored on disk. The columns are tacho, three underhang accelerometer axes, three overhang axes and the microphone.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Opens each CSV file as plain text and prints its first three lines exactly as stored, then the names of the 8 columns.</div>

In [ ]:
for r in picks:
    with open(root / r.path) as fh:
        head = [next(fh).rstrip() for _ in range(3)]
    print(f"{r.path}\n  " + "\n  ".join(head) + "\n")
print("columns:", ", ".join(pre.channels))

### Step 3 · All eight signals, first 0.2 s
0.2 s is about six shaft revolutions at 30 Hz. The tacho gives one pulse per revolution. Look for three things:
the once-per-revolution swing that imbalance adds, whether the bearing fault is visible by eye at all, and the slow
wander on the overhang channels.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the three recordings and plots all 8 channels for the first 0.2 s: one column per recording, one row per channel.</div>

In [ ]:
raw = {r.rec_id: load_csv(root / r.path) for r in picks}
n = int(0.2 * pre.fs_hz)
t_ms = np.arange(n) / pre.fs_hz * 1e3
fig, axes = plt.subplots(8, 3, figsize=(14, 12), sharex=True)
for j, r in enumerate(picks):
    x = raw[r.rec_id]
    for i, ch in enumerate(pre.channels):
        axes[i, j].plot(t_ms, x[:n, i], lw=0.5)
        if j == 0:
            axes[i, j].set_ylabel(f"{ch}\n(V)", fontsize=8)
    axes[0, j].set_title(r.rec_id, fontsize=9)
    axes[-1, j].set_xlabel("time (ms)")
fig.tight_layout()

### Step 4 · Tacho pulses and shaft speed
The speed estimator finds each pulse's rising edge (green) with hysteresis, drops double triggers, and turns the
intervals into a speed. The tacho spectrum and a harmonic comb on the radial accelerometer give two independent
cross-checks. Tacho edges win when another method confirms them within 0.2 %.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots the tachometer of each recording with the detected pulse edges in green, and computes the shaft speed with all three methods, showing which one was chosen and how closely they agree.</div>

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6.5), sharex=True)
rows = []
for ax, r in zip(axes, picks):
    x = raw[r.rec_id]
    ti = tacho_intervals(x[:, 0], pre)
    ax.plot(t_ms, x[:n, 0], lw=0.8, color="#333")
    for e in ti.edges[ti.edges < n]:
        ax.axvline(e / pre.fs_hz * 1e3, color="green", lw=1, alpha=0.7)
    ax.set(ylabel="tacho (V)", title=r.rec_id)
    cands = speed_candidates(x, labels[r.rec_id].fr_file_hz, pre)
    est = choose_speed(cands, labels[r.rec_id].fr_file_hz, pre)
    rows.append({"rec_id": r.rec_id, "nominal (Hz)": labels[r.rec_id].fr_file_hz, **{f"{k} (Hz)": v for k, v in cands.items()},
                 "chosen (Hz)": est.fr_hz, "method": est.method, "agreement (%)": est.agreement * 100})
axes[-1].set_xlabel("time (ms)")
fig.tight_layout()
pd.DataFrame(rows).set_index("rec_id").round(4)

### Step 5 · Quality numbers
For every channel: DC offset (mean), rail count (samples sitting exactly at the channel's max or min; a clean
channel has about 2) and the largest robust z-score (a spike signature). These produce the audit flags, which are
never model inputs.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes the quality numbers for each recording: its size, missing values, and for every channel the DC offset, how many samples sit at the channel's limits, and the largest spike score.</div>

In [ ]:
pd.DataFrame({r.rec_id: quality_stats(raw[r.rec_id], pre.channels) for r in picks}).round(3)

### Step 6 · The `.npy` file
`scripts/00_convert.py` stores each recording as a float32 array in `data/raw_npy/<rec_id>.npy`, so later stages
load it in milliseconds instead of parsing text. The file's sha256 must match the one recorded in `metadata.csv`,
and its values must be identical to the CSV.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Opens each converted <code>.npy</code> file and checks its shape, number type and size, that its checksum matches <code>metadata.csv</code>, and that its values are identical to the CSV.</div>

In [ ]:
for r in picks:
    path = data_dir() / "raw_npy" / f"{r.rec_id}.npy"
    arr = np.load(path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{r.rec_id}: shape {arr.shape}, {arr.dtype}, {path.stat().st_size / 1e6:.1f} MB, "
          f"sha matches metadata: {sha == r.npy_sha256}, identical to CSV: {np.array_equal(arr, raw[r.rec_id])}")

### Step 7 · The `metadata.csv` row
Everything above ends up as one row per recording: labels, the three speed estimates, quality numbers, the
conversion checksum and the splits.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows each recording's full row of <code>metadata.csv</code>, without the split columns.</div>

In [ ]:
pd.DataFrame(picks).set_index("rec_id").drop(columns=split_columns(meta)).T

### Step 8 · Split columns
Splits are by recording, assigned before any window exists, so no part of one recording can land in two splits.
Each column is a different evaluation protocol: closed-set, leave-one-class-out (one per fault class), unseen
speeds, unseen severity, and the cross-validation fold.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows each recording's speed tercile and all its split columns: which split it belongs to under each evaluation protocol.</div>

In [ ]:
pd.DataFrame(picks).set_index("rec_id")[["speed_tercile", *split_columns(meta)]].T

### Step 9 · Summary

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Prints one line per recording that summarises its whole journey from folder to split.</div>

In [ ]:
for r in picks:
    lab = labels[r.rec_id]
    print(f"{r.path} → {lab.class10} / {lab.class6}, severity {lab.severity:g} {lab.severity_unit} → "
          f"CSV {r.n_rows} × {r.n_cols} → {r.fr_measured_hz:.3f} Hz ({r.speed_method}) → "
          f"npy float32 {r.n_rows_verified} × {r.n_cols} → split_closed {r.split_closed}")

## Part B · Stage 2 (placeholders)
These steps continue the same three recordings once Stage 2 is approved. No code yet.

### Step 10 · Mic bursts removed
`ml.preprocess.remove_mic_bursts` cleans the microphone channel before any representation (CLAUDE.md §5, rule 10).

### Step 11 · Windows
Each 5 s recording is cut into windows; every window inherits its recording's labels and splits.

### Step 12 · R1 physics features
Order-domain features at the measured shaft speed: shaft harmonics for imbalance and misalignment, envelope-spectrum
bearing lines for bearing faults.

### Step 13 · Order spectrogram (R2)
The accelerometer channels resampled to shaft orders and turned into a spectrogram, plus the microphone log-Mel,
cached under the preprocessing config's sha256.

## Findings
- The three recordings go from folder to split row without loss: each `.npy` is bit-identical to its CSV and its sha256 matches `metadata.csv`.
- All three run at about 29.9 Hz measured against 30.72 Hz nominal (ratio about 0.974), and the three speed methods agree to within a few hundredths of a percent.
- Imbalance is visible by eye: a clean once-per-revolution sine on the underhang radial and tangential channels (radial std 0.62 V against 0.36 V for normal).
- The outer-race fault is not: no impacts show in the raw trace (underhang radial kurtosis 2.9, close to Gaussian). It mainly raises broadband energy, most strongly on the overhang radial channel (std 0.42 V against 0.03 V for normal). Bearing evidence has to come from the envelope spectrum.
- The overhang axial and tangential channels wander slowly by up to about 2 V within 0.2 s in the normal and imbalance recordings. This is the source of the DC offsets in the audit, and why DC is removed per window.

## Outputs written
None; this notebook only reads. It is exported to `results/00_data_journey.html`.